![image_1780244102100.png](./image_1780244102100.png "image_1780244102100.png")

![image_1780244156128.png](./image_1780244156128.png "image_1780244156128.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f  # ✅ always imported
from pyspark.sql import Window
# Initialize Spark
spark = SparkSession.builder.appName("LoginsData").getOrCreate()

# Create DataFrame
logins_data = [
    (1, "2023-01-05"),
    (2, "2023-01-10"),
    (3, "2023-01-15"),
    (4, "2023-01-20"),
    (5, "2023-01-25"),
    (1, "2023-02-01"),
    (2, "2023-02-05"),
    (3, "2023-02-10"),
    (6, "2023-02-15"),
    (7, "2023-02-20"),
    (8, "2023-02-25"),
    (1, "2023-03-01"),
    (2, "2023-03-05"),
    (9, "2023-03-10"),
    (10, "2023-03-15"),
]

logins_df = spark.createDataFrame(logins_data, ["user_id", "login_date"])

# Show DataFrame
logins_df.show()


In [0]:
result_df = (
    logins_df.withColumn("month", f.date_format("login_date", "yyyy-MM"))
    .withColumn(
        "rank",
        f.rank().over(Window.partitionBy("user_id", "month").orderBy("login_date")),
    )
    .filter(f.col("rank") == 1)
    .groupBy("month")
    .agg(f.countDistinct("user_id").alias("active_users"))
    .orderBy("month")
    .withColumn(
        "perivous_month_users", f.lag("active_users", 1).over(Window.orderBy("month"))
    )
    .select(
        f.col("month"),
        f.col("active_users"),
        f.round(
            ((f.col("active_users") - f.col("perivous_month_users")) * 100.0)
            / f.col("perivous_month_users"),
            1,
        ).alias("mom_change_pct"),
    )
)
display(result_df)